# Robust VKOSPI 기준 전략 — 실행코드 공개형 Google Colab

이 노트북은 '무SJM 거시 국면 + 균형 L2 로지스틱 + Robust VKOSPI 일간
오버레이' 기준 전략을 **압축된 코드나 프로젝트 모듈 없이** 재현합니다.
각 함수와 계산식은 아래 코드 셀에 그대로 보이며 단계별로 나눴습니다.

실행은 노트북을 Colab에서 연 뒤 '런타임 → 모두 실행'을 누르고,
업로드 창에서 robust_vkospi_colab_data.zip 하나를 선택하면 됩니다.
데이터 ZIP에는 원자료와 OAP 가공 입력만 있으며 Python 코드나 백테스트
결과는 들어 있지 않습니다. 결과는 연구용이며 투자 조언이 아닙니다.


## 0. 계산 순서

데이터 적재 → 무SJM 거시확률 → SLSQP 기본배분 → 16개 설명변수
→ 균형 L2 로지스틱 → 꼬리위험 틸트·변동성 타깃
→ Robust VKOSPI 일간 오버레이 → 성과·인과성 검증 순서입니다.


## 1. 데이터 ZIP 업로드와 실행환경 준비


In [ ]:
import json
import math
import shutil
import sqlite3
import subprocess
import sys
import tempfile
import unicodedata
import warnings
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openpyxl>=3.1"],
        check=True,
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

EXPECTED_DATA_ZIP = "robust_vkospi_colab_data.zip"


def safe_extract(archive_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"안전하지 않은 ZIP 경로: {member.filename}")
        archive.extractall(destination)


if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("데이터 ZIP 파일 하나만 업로드해 주세요.")
    DATA_ARCHIVE = Path(zip_names[0])
    RUNTIME_PARENT = Path("/content/robust_vkospi_runtime")
    if RUNTIME_PARENT.exists():
        shutil.rmtree(RUNTIME_PARENT)
    RUNTIME_PARENT.mkdir(parents=True)
else:
    DATA_ARCHIVE = Path(
        globals().get(
            "DATA_ARCHIVE_OVERRIDE",
            Path.cwd() / "artifacts/bundles" / EXPECTED_DATA_ZIP,
        )
    )
    RUNTIME_PARENT = Path(tempfile.mkdtemp(prefix="robust_vkospi_visible_"))

safe_extract(DATA_ARCHIVE, RUNTIME_PARENT)
ROOT = RUNTIME_PARENT / "RegimeDecisionData"
RAW_DIR = ROOT / "raw_data"
CACHE_DIR = ROOT / "cache"
INPUT_DIR = ROOT / "input_data"
OUTPUT_DIR = ROOT / "outputs"
VKOSPI_PATH = RAW_DIR / "VKOSPIData.csv"
OUTPUT_DIR.mkdir(exist_ok=True)
print("데이터:", DATA_ARCHIVE)
print("실행 루트:", ROOT)
print("Python:", sys.version.split()[0])


## 2. 공통 설정과 입력변수 16개

국내 포트폴리오 상태 12개와 Open Asset Pricing 기반 스트레스 합성치
4개를 월간 로지스틱 입력으로 사용합니다. VKOSPI 변수는 월간 로지스틱에
섞지 않고 마지막 일간 오버레이에서 독립적으로 사용합니다.


In [ ]:
ASSETS = ["KODEX200", "BOND", "GLD", "USO"]
DOMESTIC_FEATURES = [
    "base_USO", "base_GLD", "base_KODEX200", "p_inflation_high",
    "proxy_mom1", "proxy_mom6", "proxy_vol6",
    "daily_mom21", "daily_mom252", "daily_vol21",
    "daily_downvol21", "daily_mean_corr63",
]
OAP_COMPOSITES = [
    "oap_momentum_trend_stress",
    "oap_reversal_crowding_stress",
    "oap_low_risk_tail_stress",
    "oap_liquidity_activity_stress",
]
TAIL_FEATURES = DOMESTIC_FEATURES + OAP_COMPOSITES
RNG_SEED = 20260828

REGIME_ANCHORS = pd.DataFrame(
    {
        "Goldilocks": [0.58, 0.22, 0.15, 0.05],
        "Overheating": [0.30, 0.12, 0.23, 0.35],
        "Slowdown": [0.12, 0.66, 0.20, 0.02],
        "Stagflation": [0.08, 0.24, 0.50, 0.18],
    },
    index=ASSETS,
).T
DEFENSIVE = np.array([0.05, 0.72, 0.23, 0.00])
STRATEGIC = np.array([0.20, 0.45, 0.30, 0.05])


@dataclass(frozen=True)
class NotebookConfig:
    hard_weight: float = 0.40
    slsqp_weight: float = 0.60
    initial_leverage: float = 1.20
    tail_horizon_months: int = 2
    tail_loss_threshold: float = -0.05
    logistic_c: float = 0.10
    maximum_tail_shift: float = 0.20
    target_volatility: float = 0.15
    calibration_end: str = "2017-12"
    locked_start: str = "2018-01"


CONFIG = NotebookConfig()
display(pd.Series(asdict(CONFIG), name="value").to_frame())
display(pd.DataFrame({"input_variable": TAIL_FEATURES}))


## 3. 모듈 A — 원자료 적재

한국어 파일명은 Unicode 정규화로 찾습니다. KODEX200은 초기 KOSPI200
프록시와 실제 ETF를 이어 붙이고, GLD·USO는 USDKRW를 곱해 원화
수익률로 바꿉니다.


In [ ]:
def get_path(directory: Path, filename: str) -> Path:
    target = unicodedata.normalize("NFC", filename)
    for path in directory.iterdir():
        if unicodedata.normalize("NFC", path.name) == target:
            return path
    raise FileNotFoundError(filename)


def rolling_zscore(series: pd.Series, window: int, clip: float = 3.0) -> pd.Series:
    mean = series.rolling(window, min_periods=window).mean()
    std = series.rolling(window, min_periods=window).std(ddof=1).replace(0, np.nan)
    return ((series - mean) / std).clip(-clip, clip)


def load_macro_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    gdp = pd.read_excel(get_path(RAW_DIR, "GDP 성장률.xlsx"), index_col=0, skiprows=6)
    gdp.columns = ["QoQ", "YoY"]
    gdp.index = pd.PeriodIndex(gdp.index, freq="Q").asfreq("M", how="end").to_timestamp("M") + pd.offsets.MonthEnd(1)
    gdp = gdp.resample("ME").ffill()

    trade = pd.read_excel(get_path(RAW_DIR, "수출입 총괄_20260816.xlsx"), index_col=0, skiprows=4)
    trade = trade[["수출 금액", "수입금액"]].iloc[1:].copy()
    for col in trade.columns:
        trade[col] = trade[col].astype(str).str.replace(",", "", regex=False).astype(float)
    trade.index = pd.to_datetime(trade.index, format="%Y.%m") + pd.offsets.MonthEnd(1)
    trade["Export_YoY"] = trade["수출 금액"].pct_change(12) * 100

    bsi = pd.read_csv(get_path(RAW_DIR, "기업경기조사(전망).csv"), encoding="cp949")
    bsi = bsi[(bsi["업종코드별"] == "제 조 업") & (bsi["BSI코드별"] == "업황전망BSI 1)")]
    bsi = bsi.iloc[:, 2:4].copy()
    bsi["시점"] = bsi["시점"].str.replace("월", "", regex=False).str.replace(" ", "", regex=False)
    bsi["시점"] = pd.to_datetime(bsi["시점"], format="%Y.%m") + pd.offsets.MonthEnd(1)
    bsi = bsi.set_index("시점")
    bsi.columns = ["BSI"]

    cpi = pd.read_excel(get_path(RAW_DIR, "소비자물가 상승률.xlsx"), index_col=0, skiprows=6)
    cpi.columns = ["CPI_QoQ", "CPI_YoY"]
    cpi.index = pd.to_datetime(cpi.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    ppi = pd.read_excel(get_path(RAW_DIR, "생산자물가 상승률.xlsx"), index_col=0, skiprows=6)
    ppi.columns = ["PPI_QoQ", "PPI_YoY"]
    ppi.index = pd.to_datetime(ppi.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    prices = pd.read_excel(get_path(RAW_DIR, "수출입물가 상승률.xlsx"), index_col=0, skiprows=6)
    prices.columns = ["ExportPrice_YoY", "ImportPrice_YoY"]
    prices.index = pd.to_datetime(prices.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    core = pd.concat(
        {
            "GDP": rolling_zscore(gdp["YoY"], 72),
            "Export": rolling_zscore(trade["Export_YoY"], 36),
            "BSI": rolling_zscore(bsi["BSI"], 24),
            "CPI": rolling_zscore(cpi["CPI_YoY"], 36),
            "PPI": rolling_zscore(ppi["PPI_YoY"], 36),
            "ImportPrice": rolling_zscore(prices["ImportPrice_YoY"], 36),
        },
        axis=1,
    ).sort_index()

    # Levels describe the phase; 3-month changes help identify turning points.
    growth = core[["GDP", "Export", "BSI"]].copy()
    growth.columns = ["GDP_level", "Export_level", "BSI_level"]
    growth = pd.concat([growth, growth.diff(3).add_suffix("_d3")], axis=1)
    inflation = core[["CPI", "PPI", "ImportPrice"]].copy()
    inflation.columns = ["CPI_level", "PPI_level", "ImportPrice_level"]
    inflation = pd.concat([inflation, inflation.diff(3).add_suffix("_d3")], axis=1)
    features = pd.concat({"growth": growth, "inflation": inflation}, axis=1).dropna()
    return features, core


def download_market_cache(refresh: bool = False) -> pd.DataFrame:
    cache = CACHE_DIR / "market_daily.csv"
    if cache.exists() and not refresh:
        out = pd.read_csv(cache, parse_dates=["date"])
        return out

    import yfinance as yf

    rows: list[pd.DataFrame] = []
    for ticker, symbol in [("069500.KS", "KODEX200"), ("GLD", "GLD"), ("USO", "USO"), ("KRW=X", "USDKRW")]:
        data = yf.download(ticker, start="2000-01-01", auto_adjust=(symbol != "USDKRW"), progress=False, threads=False)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        data = data.reset_index().rename(columns={"Date": "date", "Open": "open", "Close": "close"})
        data["date"] = pd.to_datetime(data["date"], utc=True).dt.tz_localize(None).dt.normalize()
        data["symbol"] = symbol
        rows.append(data[["date", "symbol", "open", "close"]])
    out = pd.concat(rows, ignore_index=True).dropna(subset=["date", "close"])
    out.to_csv(cache, index=False)
    return out


def load_monthly_asset_returns(refresh: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    market = download_market_cache(refresh)

    with sqlite3.connect(get_path(RAW_DIR, "compass.db")) as con:
        proxy = pd.read_sql(
            "select date, open, close from etf_prices where symbol = ? order by date",
            con,
            params=("1028",),
        )
    proxy["date"] = pd.to_datetime(proxy["date"])
    proxy[["open", "close"]] = proxy[["open", "close"]].apply(pd.to_numeric, errors="coerce")

    actual = market[market["symbol"] == "KODEX200"].copy().dropna(subset=["open"])
    # Yahoo contains a sparse early fragment followed by a long gap.  Use the
    # continuous KOSPI200 proxy through March 2009, then splice the ETF series.
    actual = actual[actual["date"] > pd.Timestamp("2009-03-31")]
    first_actual = actual["date"].min()
    actual_anchor = actual.loc[actual["date"] == first_actual, "open"].iloc[0]
    proxy_anchor = proxy.loc[proxy["date"] == first_actual, "open"]
    if proxy_anchor.empty:
        nearest = proxy.iloc[(proxy["date"] - first_actual).abs().argsort()[:1]]
        proxy_anchor_value = float(nearest["open"].iloc[0])
    else:
        proxy_anchor_value = float(proxy_anchor.iloc[0])
    proxy["open"] = proxy["open"] * float(actual_anchor) / proxy_anchor_value
    proxy["close"] = proxy["close"] * float(actual_anchor) / proxy_anchor_value
    proxy = proxy[proxy["date"] < first_actual]
    proxy["symbol"] = "KODEX200"
    kodex = pd.concat([proxy[["date", "symbol", "open", "close"]], actual], ignore_index=True)

    bond = pd.read_csv(get_path(RAW_DIR, "krx_bond_index.csv"), encoding="cp949")
    bond["date"] = pd.to_datetime(bond.iloc[:, 0])
    bond["open"] = bond.iloc[:, 1].astype(str).str.replace(",", "", regex=False).astype(float)
    bond["close"] = bond["open"]
    bond["symbol"] = "BOND"

    fx = market[market["symbol"] == "USDKRW"].set_index("date")["close"].sort_index()
    fx = fx.reindex(pd.date_range(fx.index.min(), fx.index.max(), freq="D")).ffill()

    first_open: dict[str, pd.Series] = {}
    trade_dates: dict[str, pd.Series] = {}
    for symbol, data in {
        "KODEX200": kodex,
        "BOND": bond,
        "GLD": market[market["symbol"] == "GLD"],
        "USO": market[market["symbol"] == "USO"],
    }.items():
        temp = data.dropna(subset=["open"]).sort_values("date").copy()
        temp["month"] = temp["date"].dt.to_period("M")
        first = temp.groupby("month", sort=True).first()
        value = first["open"].astype(float)
        if symbol in {"GLD", "USO"}:
            value = value * fx.reindex(pd.DatetimeIndex(first["date"]), method="ffill").to_numpy()
        first_open[symbol] = value
        trade_dates[symbol] = first["date"]

    levels = pd.concat(first_open, axis=1).sort_index()
    returns = levels.shift(-1).div(levels).sub(1.0).dropna(how="any")
    returns = returns[ASSETS]
    return returns, levels[ASSETS]


In [ ]:
monthly_returns, monthly_levels = load_monthly_asset_returns(False)
macro_features, macro_core = load_macro_data()
print("월별 수익률:", monthly_returns.index.min(), "→", monthly_returns.index.max())
print("거시 특징:", macro_features.index.min().date(), "→", macro_features.index.max().date())
display(monthly_returns.head())


## 4. 모듈 B — SLSQP 위험제어 배분

합계 100%, 자산별 상·하한, 목표 변동성, CDaR 제약을 두고 기대수익,
변동성, 꼬리손실, 회전율, 국면 기준점 이탈을 목적함수에서 절충합니다.
아래 셀에 목적함수와 제약식 전체가 있습니다.


In [ ]:
def soft_anchor(signal: pd.Series) -> np.ndarray:
    p = np.array([signal[f"p_{r}"] for r in REGIME_ANCHORS.index])
    return p @ REGIME_ANCHORS.to_numpy()


def ewma_cov(history: pd.DataFrame, half_life: float = 12.0, leverage: float = 1.0) -> np.ndarray:
    x = history[ASSETS].to_numpy(dtype=float)
    if len(x) < 12:
        return np.cov(x, rowvar=False) + np.eye(len(ASSETS)) * 1e-6
    alpha = 1 - math.exp(math.log(0.5) / half_life)
    cov = np.cov(x[: min(24, len(x))], rowvar=False)
    mean = np.nanmean(x, axis=0)
    for row in x:
        shock = row - mean
        multiplier = 1.0 + leverage * min(max(-row[0], 0.0) / 0.08, 1.5)
        cov = (1 - alpha) * cov + alpha * multiplier * np.outer(shock, shock)
    return cov + np.eye(len(ASSETS)) * 1e-7


def cdar(returns: np.ndarray, alpha: float = 0.90) -> float:
    wealth = np.cumprod(1 + returns)
    dd = wealth / np.maximum.accumulate(np.r_[1.0, wealth])[-len(wealth):] - 1.0
    k = max(1, int(math.ceil((1 - alpha) * len(dd))))
    return float(np.mean(np.sort(dd)[:k]))


@dataclass
class StrategyConfig:
    name: str = "Proposed"
    target_vol: float = 0.08
    half_life: float = 12.0
    invvol_tilt: float = 0.35
    return_reward: float = 1.15
    vol_penalty: float = 0.18
    cdar_penalty: float = 0.25
    turnover_penalty: float = 0.05
    tracking_penalty: float = 0.32
    max_cdar: float = 0.16
    drawdown_guard: float = 0.75
    regime_strength: float = 0.75
    use_regime: bool = True
    use_risk_control: bool = True


def controlled_weights(
    signal: pd.Series,
    history: pd.DataFrame,
    pretrade: np.ndarray,
    current_dd: float,
    cfg: StrategyConfig,
) -> np.ndarray:
    anchor = soft_anchor(signal) if cfg.use_regime else STRATEGIC.copy()
    anchor = cfg.regime_strength * anchor + (1 - cfg.regime_strength) * STRATEGIC
    if not cfg.use_risk_control or len(history) < 24:
        return anchor

    cov = ewma_cov(history.tail(84), cfg.half_life, leverage=1.0)
    vols = np.sqrt(np.diag(cov)).clip(0.005, None)
    tilted = anchor * (np.median(vols) / vols) ** cfg.invvol_tilt
    tilted = tilted / tilted.sum()
    prior = 0.55 * anchor + 0.45 * tilted

    hist = history.tail(84)[ASSETS]
    long_mu = history[ASSETS].expanding(min_periods=24).mean().iloc[-1].to_numpy()
    recent_mu = hist.ewm(halflife=24, adjust=False).mean().iloc[-1].to_numpy()
    mu = 0.80 * long_mu + 0.20 * recent_mu
    mu = np.clip(mu, -0.006, 0.015)
    target = cfg.target_vol * (0.86 + 0.20 * float(signal["p_growth_high"]))

    def objective(w: np.ndarray) -> float:
        ann_return = 12 * float(w @ mu)
        ann_vol = math.sqrt(max(float(w @ cov @ w), 0.0) * 12)
        path_cdar = abs(cdar(hist.to_numpy() @ w, 0.90))
        turnover = 0.5 * np.sum(np.sqrt((w - pretrade) ** 2 + 1e-6))
        tracking = float(np.sum((w - prior) ** 2))
        return (
            -cfg.return_reward * ann_return
            + cfg.vol_penalty * ann_vol
            + cfg.cdar_penalty * path_cdar
            + cfg.turnover_penalty * turnover
            + cfg.tracking_penalty * tracking
        )

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "ineq", "fun": lambda w: target - math.sqrt(max(float(w @ cov @ w), 0.0) * 12)},
        {"type": "ineq", "fun": lambda w: cfg.max_cdar + cdar(hist.to_numpy() @ w, 0.90)},
    ]
    bounds = [(0.02, 0.68), (0.05, 0.88), (0.02, 0.62), (0.0, 0.38)]
    result = minimize(objective, prior, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 80, "ftol": 1e-8})
    w = result.x if result.success and np.isfinite(result.x).all() else prior
    w = np.clip(w, 0, None)
    w /= w.sum()

    # MPC-style state-dependent risk aversion: react to realized drawdown, but
    # retain a floor in risky assets so recovery participation is not lost.
    if current_dd < -0.05:
        severity = min(max((-current_dd - 0.05) / 0.12, 0.0), 1.0)
        blend = cfg.drawdown_guard * (0.25 + 0.50 * severity)
        w = (1 - blend) * w + blend * DEFENSIVE
    return w / w.sum()


def hard_regime_weights(signal: pd.Series) -> np.ndarray:
    mapping = {
        "Goldilocks": np.array([1.0, 0.0, 0.0, 0.0]),
        "Overheating": np.array([0.0, 0.0, 0.0, 1.0]),
        "Slowdown": np.array([0.6, 0.4, 0.0, 0.0]),
        "Stagflation": np.array([0.0, 0.0, 1.0, 0.0]),
    }
    return mapping[signal["regime"]]


def run_backtest(
    returns: pd.DataFrame,
    signals: pd.DataFrame,
    cfg: StrategyConfig,
    mode: str = "proposed",
    start: str | None = None,
    end: str | None = None,
    cost_multiplier: float = 1.0,
) -> pd.DataFrame:
    months = signals.index.intersection(returns.index)
    if start:
        months = months[months >= pd.Period(start, "M")]
    if end:
        months = months[months <= pd.Period(end, "M")]
    rows = []
    pretrade = np.zeros(4)
    first_trade = True
    nav = 1.0
    peak = 1.0
    for month in months:
        signal = signals.loc[month]
        history = returns.loc[returns.index < month]
        current_dd = nav / peak - 1.0
        if mode == "proposed":
            w = controlled_weights(signal, history, pretrade, current_dd, cfg)
        elif mode == "soft":
            w = soft_anchor(signal)
        elif mode == "hard":
            w = hard_regime_weights(signal)
        elif mode == "equal":
            w = np.full(4, 0.25)
        elif mode == "static_defensive":
            w = np.array([0.20, 0.45, 0.30, 0.05])
        elif mode == "kodex":
            w = np.array([1.0, 0.0, 0.0, 0.0])
        else:
            raise ValueError(mode)

        delta = w - pretrade
        turnover = np.abs(delta).sum() if first_trade else 0.5 * np.abs(delta).sum()
        trade_cost = np.abs(delta).sum() * 0.0015 * cost_multiplier
        fx_cost = abs((w[2] + w[3]) - (pretrade[2] + pretrade[3])) * 0.0005 * cost_multiplier
        gross_return = float(w @ returns.loc[month, ASSETS].to_numpy())
        net_return = gross_return - trade_cost - fx_cost
        nav *= 1 + net_return
        peak = max(peak, nav)
        end_w = w * (1 + returns.loc[month, ASSETS].to_numpy()) / (1 + gross_return)
        rows.append(
            {
                "month": month,
                "signal_month": signal["signal_month"],
                "regime": signal["regime"],
                "p_growth_high": signal["p_growth_high"],
                "p_inflation_high": signal["p_inflation_high"],
                "gross_return": gross_return,
                "return": net_return,
                "turnover": turnover,
                "trade_cost": trade_cost,
                "fx_cost": fx_cost,
                "nav": nav,
                "drawdown": nav / peak - 1,
                **{f"w_{a}": w[i] for i, a in enumerate(ASSETS)},
            }
        )
        pretrade = end_w
        first_trade = False
    return pd.DataFrame(rows).set_index("month")


def performance_summary(returns: pd.Series) -> pd.Series:
    r = pd.Series(returns).dropna()
    wealth = (1 + r).cumprod()
    years = len(r) / 12
    cagr = wealth.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    vol = r.std(ddof=1) * math.sqrt(12)
    sharpe = r.mean() / r.std(ddof=1) * math.sqrt(12) if r.std(ddof=1) > 0 else np.nan
    dd = wealth / wealth.cummax() - 1
    mdd = dd.min()
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan
    downside = np.sqrt(np.mean(np.minimum(r, 0) ** 2)) * math.sqrt(12)
    sortino = r.mean() * 12 / downside if downside > 0 else np.nan
    return pd.Series(
        {
            "Months": len(r),
            "CAGR": cagr,
            "Volatility": vol,
            "Sharpe": sharpe,
            "Sortino": sortino,
            "MDD": mdd,
            "Calmar": calmar,
            "FinalMultiple": wealth.iloc[-1],
            "PositiveMonths": (r > 0).mean(),
        }
    )


## 5. 모듈 C — 무SJM 거시국면

성장축은 GDP·수출·BSI, 물가축은 CPI·PPI·수입물가의 표준화 수준과
3개월 변화를 사용합니다. SJM 가중치는 0입니다. 현재 확률 85%와
직전 확률 15%를 섞어 월간 급변을 완화합니다.


In [ ]:
def _macro_probabilities(
    components: pd.DataFrame,
    d3_weight: float,
    sigmoid_scale: float,
    sjm_weight: float,
    current_weight: float,
) -> pd.DataFrame:
    output = pd.DataFrame(index=components.index, dtype=float)
    previous = {"growth": 0.5, "inflation": 0.5}
    for target_month, row in components.iterrows():
        for name in ("growth", "inflation"):
            composite = float(
                expit((row[f"{name}_level"] + d3_weight * row[f"{name}_d3"]) / sigmoid_scale)
            )
            raw = sjm_weight * row[f"{name}_sjm"] + (1 - sjm_weight) * composite
            probability = current_weight * raw + (1 - current_weight) * previous[name]
            output.loc[target_month, name] = probability
            previous[name] = probability
    return output


def build_no_sjm_components(returns: pd.DataFrame) -> pd.DataFrame:
    """Build the causal macro component panel shared by no-SJM variants."""
    macro, _ = load_macro_data()
    rows: list[dict[str, float | pd.Period]] = []
    for target_month in returns.index:
        signal_month = target_month - 1
        history = macro.loc[: signal_month.to_timestamp("M")]
        if len(history) < 24:
            continue
        features = history.iloc[-1]
        rows.append(
            {
                "target_month": target_month,
                "signal_month": signal_month,
                "growth_level": float(
                    features["growth"][["GDP_level", "Export_level", "BSI_level"]].mean()
                ),
                "growth_d3": float(
                    features["growth"][
                        ["GDP_level_d3", "Export_level_d3", "BSI_level_d3"]
                    ].mean()
                ),
                "inflation_level": float(
                    features["inflation"][
                        ["CPI_level", "PPI_level", "ImportPrice_level"]
                    ].mean()
                ),
                "inflation_d3": float(
                    features["inflation"][
                        ["CPI_level_d3", "PPI_level_d3", "ImportPrice_level_d3"]
                    ].mean()
                ),
                # The probability function has one shared interface. These
                # placeholders are deliberately ignored when sjm_weight=0.
                "growth_sjm": 0.5,
                "inflation_sjm": 0.5,
            }
        )
    components = pd.DataFrame(rows).set_index("target_month")
    components.index = pd.PeriodIndex(components.index, freq="M")
    return components


def build_macro_signals(
    probabilities: pd.DataFrame,
    components: pd.DataFrame,
) -> pd.DataFrame:
    output = pd.DataFrame(index=probabilities.index)
    output["signal_month"] = components["signal_month"]
    output["p_growth_high"] = probabilities["growth"]
    output["p_inflation_high"] = probabilities["inflation"]
    output["p_Goldilocks"] = probabilities["growth"] * (1 - probabilities["inflation"])
    output["p_Overheating"] = probabilities["growth"] * probabilities["inflation"]
    output["p_Slowdown"] = (1 - probabilities["growth"]) * (1 - probabilities["inflation"])
    output["p_Stagflation"] = (1 - probabilities["growth"]) * probabilities["inflation"]
    regime_columns = [
        "p_Goldilocks",
        "p_Overheating",
        "p_Slowdown",
        "p_Stagflation",
    ]
    output["regime"] = output[regime_columns].idxmax(axis=1).str.removeprefix("p_")
    return output


In [ ]:
def build_no_sjm_signals(returns: pd.DataFrame):
    components = build_no_sjm_components(returns)
    probabilities = _macro_probabilities(
        components, d3_weight=0.20, sigmoid_scale=0.55,
        sjm_weight=0.0, current_weight=0.85,
    )
    signals = build_macro_signals(probabilities, components)
    assert (signals["signal_month"] < signals.index).all()
    assert np.isfinite(probabilities.to_numpy(dtype=float)).all()
    return signals, probabilities


signals, macro_probabilities = build_no_sjm_signals(monthly_returns)
defensive = run_backtest(
    monthly_returns, signals, StrategyConfig(), mode="proposed"
)
print("첫 거시 예측 대상월:", signals.index.min())
display(signals.head())


## 6. 모듈 D — 16개 입력과 꼬리사건 라벨

40% hard 국면과 60% SLSQP를 섞고 1.2배 노출을 적용한 중립경로를
만듭니다. 그 경로의 앞으로 2개월 진행 중 누적손실이 -5%보다 작으면
꼬리사건입니다. 마지막 두 달은 미래가 완성되지 않아 결측입니다.


In [ ]:
def _daily_asset_returns() -> pd.DataFrame:
    market = pd.read_csv(ROOT / "cache" / "market_daily.csv", parse_dates=["date"])
    levels = market.pivot_table(
        index="date", columns="symbol", values="close", aggfunc="last"
    ).sort_index()
    levels["GLD"] = levels["GLD"] * levels["USDKRW"]
    levels["USO"] = levels["USO"] * levels["USDKRW"]
    bond = pd.read_csv(ROOT / "raw_data" / "krx_bond_index.csv", encoding="cp949")
    bond.index = pd.to_datetime(bond.iloc[:, 0])
    bond_level = pd.to_numeric(
        bond.iloc[:, 1].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )
    daily_levels = pd.concat(
        [
            levels["KODEX200"],
            bond_level.rename("BOND"),
            levels["GLD"],
            levels["USO"],
        ],
        axis=1,
    ).sort_index()
    daily_levels.columns = ASSETS
    daily_levels = daily_levels.reindex(
        pd.date_range(daily_levels.index.min(), daily_levels.index.max(), freq="B")
    ).ffill(limit=5)
    return daily_levels.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)


def build_domestic_features(
    signals: pd.DataFrame,
    returns: pd.DataFrame,
) -> pd.DataFrame:
    """Rebuild the 12 deployed domestic inputs under the no-SJM regime path."""
    daily_returns = _daily_asset_returns()
    rows: list[dict[str, object]] = []
    for month in signals.index.intersection(returns.index):
        history = returns.loc[returns.index < month, ASSETS]
        if len(history) < 3:
            continue
        signal = signals.loc[month]
        base = hard_regime_weights(signal)
        proxy = pd.Series(history.to_numpy(dtype=float) @ base, index=history.index)
        cutoff = month.to_timestamp(how="start") - pd.Timedelta(days=2)
        daily_history = daily_returns.loc[:cutoff, ASSETS].dropna(how="all")
        daily_proxy = pd.Series(
            daily_history.fillna(0.0).to_numpy(dtype=float) @ base,
            index=daily_history.index,
        )

        row: dict[str, object] = {
            "month": month,
            "base_KODEX200": float(base[0]),
            "base_BOND": float(base[1]),
            "base_GLD": float(base[2]),
            "base_USO": float(base[3]),
            "p_inflation_high": float(signal["p_inflation_high"]),
            "proxy_mom1": float((1 + proxy.tail(1)).prod() - 1),
            "proxy_mom6": float((1 + proxy.tail(6)).prod() - 1),
            "proxy_vol6": float(proxy.tail(6).std(ddof=1) * math.sqrt(12)),
            "daily_mom21": float((1 + daily_proxy.tail(21)).prod() - 1),
            "daily_mom252": float((1 + daily_proxy.tail(252)).prod() - 1),
            "daily_vol21": float(daily_proxy.tail(21).std(ddof=1) * math.sqrt(252)),
            "daily_downvol21": float(
                np.sqrt(np.mean(np.minimum(daily_proxy.tail(21), 0.0) ** 2))
                * math.sqrt(252)
            ),
        }
        correlation = daily_history.tail(63)[["KODEX200", "GLD", "USO"]].corr()
        values = correlation.to_numpy()[np.triu_indices(3, 1)]
        row["daily_mean_corr63"] = float(np.nanmean(values))
        rows.append(row)

    output = pd.DataFrame(rows).set_index("month")
    output.index = pd.PeriodIndex(output.index, freq="M")
    missing = [column for column in DOMESTIC_FEATURES if column not in output]
    if missing:
        raise ValueError(f"Missing domestic inputs: {missing}")
    return output


def run_neutral_factor_blend(
    returns: pd.DataFrame,
    signals: pd.DataFrame,
    defensive: pd.DataFrame,
    cost_multiplier: float = 1.0,
) -> pd.DataFrame:
    """Reproduce the fixed 40% hard + 60% SLSQP, 1.2x baseline used by the label."""
    months = signals.index.intersection(returns.index).intersection(defensive.index)
    rows: list[dict[str, object]] = []
    pretrade = np.zeros(len(ASSETS))
    first_trade = True
    nav = 1.0
    peak = 1.0
    for month in months:
        hard = hard_regime_weights(signals.loc[month])
        defensive_weights = defensive.loc[
            month, [f"w_{asset}" for asset in ASSETS]
        ].to_numpy(dtype=float)
        unlevered = 0.40 * hard + 0.60 * defensive_weights
        weights = 1.20 * unlevered
        debt_weight = -0.20
        delta = weights - pretrade
        turnover = np.abs(delta).sum() if first_trade else 0.5 * np.abs(delta).sum()
        trade_cost = np.abs(delta).sum() * 0.0015 * cost_multiplier
        fx_cost = (
            abs((weights[2] + weights[3]) - (pretrade[2] + pretrade[3]))
            * 0.0005
            * cost_multiplier
        )
        financing = debt_weight * ((1 + 0.04) ** (1 / 12) - 1)
        asset_return = returns.loc[month, ASSETS].to_numpy(dtype=float)
        gross_return = float(weights @ asset_return + financing)
        net_return = gross_return - trade_cost - fx_cost
        nav *= 1 + net_return
        peak = max(peak, nav)
        pretrade = weights * (1 + asset_return) / (1 + gross_return)
        first_trade = False
        rows.append(
            {
                "month": month,
                "return": net_return,
                "gross_return": gross_return,
                "nav": nav,
                "drawdown": nav / peak - 1,
                "turnover": float(turnover),
                "trade_cost": float(trade_cost),
                "fx_cost": float(fx_cost),
                **{f"w_{asset}": float(weights[index]) for index, asset in enumerate(ASSETS)},
            }
        )
    return pd.DataFrame(rows).set_index("month")


def forward_path_loss(returns: pd.Series, horizon: int = 2) -> pd.Series:
    output = pd.Series(np.nan, index=returns.index, dtype=float)
    for position in range(len(returns)):
        future = returns.iloc[position : position + horizon]
        if len(future) == horizon:
            output.iloc[position] = float(((1 + future).cumprod() - 1).min())
    return output


def balanced_logistic_spec() -> dict[str, object]:
    return {
        "candidate": "l2_liblinear_c0.1_balanced",
        "penalty": "l2",
        "solver": "liblinear",
        "C": 0.10,
        "class_weight": "balanced",
        "l1_ratio": np.nan,
    }


In [ ]:
domestic_features = build_domestic_features(signals, monthly_returns)
oap = pd.read_csv(
    INPUT_DIR / "openassetpricing_composites.csv", index_col=0
)
oap.index = pd.PeriodIndex(oap.index, freq="M")

neutral_baseline = run_neutral_factor_blend(
    monthly_returns, signals, defensive
)
model_data = domestic_features[DOMESTIC_FEATURES].join(
    oap[OAP_COMPOSITES], how="left"
)
model_data = model_data.loc[
    model_data.index.intersection(neutral_baseline.index)
].copy()
path_loss = forward_path_loss(
    neutral_baseline.loc[model_data.index, "return"],
    horizon=CONFIG.tail_horizon_months,
)
model_data["tail_event"] = (
    path_loss < CONFIG.tail_loss_threshold
).where(path_loss.notna()).astype(float)

print("학습 패널:", model_data.shape)
print("꼬리사건 수:", int(model_data["tail_event"].sum()))
display(model_data.tail())


## 7. 모듈 E — 균형 L2 로지스틱 워크포워드

매월 모델을 새로 학습합니다. 해당 월에서 두 달을 비우는 embargo를
두고 그보다 과거만 사용합니다. 최소 36개 행, 양성 4개, 음성 12개가
모이기 전에는 확률을 만들지 않습니다. 결측치 중앙값 대체와 표준화도
매 시점의 학습표본에서 다시 적합합니다. class_weight는 balanced,
L2 규제 C는 0.1, solver는 liblinear입니다.


In [ ]:
def tail_causal_percentile(values: pd.Series, lookback: int = 60) -> pd.Series:
    """Map a value to its trailing empirical percentile without using the current row."""
    result = pd.Series(np.nan, index=values.index, dtype=float)
    history: list[float] = []
    for index, value in values.items():
        if not np.isfinite(value):
            continue
        reference = np.asarray(history[-lookback:], dtype=float)
        if len(reference) >= 12:
            result.loc[index] = (
                float(np.sum(reference <= value)) + 1.0
            ) / (len(reference) + 1.0)
        else:
            result.loc[index] = 0.5
        history.append(float(value))
    return result


def _transfer(
    weights: np.ndarray,
    donors: list[int],
    receivers: list[int],
    amount: float,
) -> np.ndarray:
    """Copy of the deployed factor-tilt transfer, kept local for Colab portability."""
    output = weights.copy()
    floors = np.array([0.02, 0.05, 0.02, 0.00])
    available = np.maximum(output[donors] - floors[donors], 0)
    transfer = min(float(amount), float(available.sum()))
    if transfer <= 0:
        return output
    output[donors] -= transfer * available / available.sum()
    receiver_base = np.maximum(output[receivers], 0.02)
    output[receivers] += transfer * receiver_base / receiver_base.sum()
    return output / output.sum()


def apply_factor_tilt(base: np.ndarray, score: float, max_shift: float) -> np.ndarray:
    if not np.isfinite(score) or max_shift <= 0:
        return base.copy()
    if score >= 0:
        return _transfer(base, donors=[1, 2], receivers=[0], amount=max_shift * score)
    return _transfer(base, donors=[0, 3], receivers=[1, 2], amount=max_shift * -score)


def run_factor_vol_target(
    returns: pd.DataFrame,
    signals: pd.DataFrame,
    defensive_path: pd.DataFrame,
    factor: pd.DataFrame,
    max_shift: float,
    target_vol: float = 0.15,
    cost_multiplier: float = 1.0,
    financing_rate: float = 0.04,
    leverage_min: float = 0.50,
    leverage_max: float = 1.50,
    initial_leverage: float = 1.20,
) -> pd.DataFrame:
    """Reproduce the deployed medium-horizon allocation without heavy optional imports."""
    if not 0 < leverage_min <= leverage_max:
        raise ValueError("leverage bounds must satisfy 0 < minimum <= maximum")
    initial_leverage = float(np.clip(initial_leverage, leverage_min, leverage_max))
    months = signals.index.intersection(returns.index).intersection(defensive_path.index)
    rows: list[dict[str, float | pd.Period | str]] = []
    pretrade = np.zeros(len(ASSETS))
    nav = 1.0
    peak = 1.0
    first_trade = True
    for month in months:
        hard = hard_regime_weights(signals.loc[month])
        defensive = defensive_path.loc[
            month, [f"w_{asset}" for asset in ASSETS]
        ].to_numpy(dtype=float)
        base = 0.40 * hard + 0.60 * defensive
        probability = float(factor.loc[month, "p_up"]) if month in factor.index else 0.5
        score = float(np.clip((probability - 0.5) / 0.15, -1, 1))
        unlevered = apply_factor_tilt(base, score, max_shift)

        history = returns.loc[returns.index < month, ASSETS].tail(24)
        if len(history) >= 12:
            conditional = history.to_numpy(dtype=float) @ unlevered
            observation_weights = np.exp(np.linspace(-2.0, 0.0, len(conditional)))
            observation_weights /= observation_weights.sum()
            mean = float(observation_weights @ conditional)
            variance = float(observation_weights @ (conditional - mean) ** 2)
            forecast_vol = math.sqrt(max(variance, 1e-8) * 12)
            leverage = float(
                np.clip(target_vol / forecast_vol, leverage_min, leverage_max)
            )
        else:
            forecast_vol = np.nan
            leverage = initial_leverage
        asset_weights = leverage * unlevered
        debt_weight = 1.0 - leverage
        delta = asset_weights - pretrade
        turnover = np.abs(delta).sum() if first_trade else 0.5 * np.abs(delta).sum()
        trade_cost = np.abs(delta).sum() * 0.0015 * cost_multiplier
        fx_cost = (
            abs((asset_weights[2] + asset_weights[3]) - (pretrade[2] + pretrade[3]))
            * 0.0005
            * cost_multiplier
        )
        financing = debt_weight * ((1 + financing_rate) ** (1 / 12) - 1)
        asset_return = returns.loc[month, ASSETS].to_numpy(dtype=float)
        gross_return = float(asset_weights @ asset_return + financing)
        net_return = gross_return - trade_cost - fx_cost
        nav *= 1 + net_return
        peak = max(peak, nav)
        pretrade = asset_weights * (1 + asset_return) / (1 + gross_return)
        first_trade = False
        rows.append(
            {
                "month": month,
                "return": net_return,
                "gross_return": gross_return,
                "nav": nav,
                "drawdown": nav / peak - 1,
                "turnover": turnover,
                "trade_cost": trade_cost,
                "fx_cost": fx_cost,
                "forecast_vol": forecast_vol,
                "leverage": leverage,
                "risk_score": score,
                **{
                    f"w_{asset}": asset_weights[index]
                    for index, asset in enumerate(ASSETS)
                },
            }
        )
    return pd.DataFrame(rows).set_index("month")


def make_logistic_model(spec: dict[str, object]) -> Pipeline:
    arguments: dict[str, object] = {
        "C": float(spec["C"]),
        "penalty": str(spec["penalty"]),
        "solver": str(spec["solver"]),
        "class_weight": spec["class_weight"],
        "max_iter": 5000,
        "random_state": RNG_SEED,
    }
    if spec["penalty"] == "elasticnet":
        arguments["l1_ratio"] = float(spec["l1_ratio"])
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(**arguments)),
        ]
    )


def fit_logistic_candidate(
    data: pd.DataFrame,
    spec: dict[str, object],
) -> tuple[pd.Series, dict[str, float]]:
    probability = pd.Series(np.nan, index=data.index, dtype=float)
    l1_norms: list[float] = []
    l2_norms: list[float] = []
    nonzero_counts: list[int] = []
    convergence_warnings = 0
    for number, month in enumerate(data.index):
        train_end = number - 2
        if train_end < 36:
            continue
        train = data.iloc[:train_end].dropna(subset=["tail_event"])
        y = train["tail_event"].astype(int)
        if y.sum() < 4 or (len(y) - y.sum()) < 12:
            continue
        model = make_logistic_model(spec)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(train[TAIL_FEATURES], y)
        convergence_warnings += sum(
            issubclass(item.category, ConvergenceWarning) for item in caught
        )
        probability.loc[month] = float(
            model.predict_proba(data.loc[[month], TAIL_FEATURES])[:, 1][0]
        )
        coefficient = model.named_steps["model"].coef_[0]
        l1_norms.append(float(np.abs(coefficient).sum()))
        l2_norms.append(float(np.sqrt(np.square(coefficient).sum())))
        nonzero_counts.append(int(np.sum(np.abs(coefficient) > 1e-8)))
    return probability, {
        "fit_count": float(len(l1_norms)),
        "convergence_warning_count": float(convergence_warnings),
        "median_coefficient_l1_norm": float(np.median(l1_norms)),
        "median_coefficient_l2_norm": float(np.median(l2_norms)),
        "median_nonzero_coefficients": float(np.median(nonzero_counts)),
    }


def make_tail_factor(probability: pd.Series, target: pd.Series) -> pd.DataFrame:
    output = pd.DataFrame({"p_tail_raw": probability, "tail_event": target})
    output["risk_percentile"] = tail_causal_percentile(output["p_tail_raw"])
    output["risk_severity"] = (
        (output["risk_percentile"] - 0.80) / 0.20
    ).clip(0, 1)
    output["p_up"] = 0.50 - 0.15 * output["risk_severity"]
    output["score"] = -output["risk_severity"]
    return output


In [ ]:
probability, fit_stats = fit_logistic_candidate(
    model_data, balanced_logistic_spec()
)
factor = make_tail_factor(probability, model_data["tail_event"])

prediction_view = factor.dropna(
    subset=["p_tail_raw", "tail_event"]
)
y = prediction_view["tail_event"].astype(int)
p = prediction_view["p_tail_raw"].clip(1e-6, 1 - 1e-6)
prediction_scores = pd.Series(
    {
        "observations": len(prediction_view),
        "events": int(y.sum()),
        "ROC_AUC": roc_auc_score(y, p),
        "AveragePrecision": average_precision_score(y, p),
        "BrierScore": brier_score_loss(y, p),
        **fit_stats,
    }
)
print("첫 유효 로지스틱 예측월:", factor["p_tail_raw"].first_valid_index())
display(prediction_scores.to_frame("value"))
display(factor.tail(12))


## 8. 모듈 F — 꼬리위험 틸트와 15% 변동성 타깃

최근 60개 유효 확률 안에서 현재 확률의 인과적 백분위를 계산합니다.
80백분위 이하에서는 이동하지 않고 80~100백분위에서 위험강도가
0에서 1로 커집니다. 최대 20%를 주식·원유에서 채권·금으로 옮긴 뒤,
최근 24개월 가중 변동성으로 총 노출을 0.5~1.5배에서 조정합니다.


In [ ]:
medium = run_factor_vol_target(
    monthly_returns,
    signals,
    defensive,
    factor,
    max_shift=CONFIG.maximum_tail_shift,
    target_vol=CONFIG.target_volatility,
)
display(medium[[
    "return", "leverage", "risk_score",
    "w_KODEX200", "w_BOND", "w_GLD", "w_USO",
]].tail())


## 9. 모듈 G — 일간 자산·VKOSPI 정렬과 실행

월간 기준비중을 각 영업일에 펼치고 다음 영업일 시가 수익률에
적용합니다. VKOSPI 신호는 행동일 전일까지 잘라 사용합니다. 월초에는
리밸런싱하고 월중 목표와 현재 비중 차이의 절반합이 20% 이상일 때만
거래해 회전율을 억제합니다.


In [ ]:
@dataclass(frozen=True)
class DynamicRiskConfig:
    mode: str
    level_threshold: float
    momentum_window: int
    spike_threshold: float
    max_risk_transfer: float
    bond_share: float = 0.50
    rebalance_band: float = 0.05
    financing_rate: float = 0.04

    @property
    def name(self) -> str:
        return (
            f"{self.mode}_lt{self.level_threshold:.2f}_mw{self.momentum_window}"
            f"_st{self.spike_threshold:.2f}_rt{self.max_risk_transfer:.2f}"
            f"_bs{self.bond_share:.2f}_rb{self.rebalance_band:.2f}"
        )


def load_vkospi_daily(path: Path = VKOSPI_PATH) -> pd.DataFrame:
    raw = pd.read_csv(path, encoding="utf-8-sig")
    if raw.shape[1] < 7:
        raise ValueError(f"VKOSPI file must contain seven columns: {path}")
    daily = raw.iloc[:, :7].copy()
    daily.columns = ["date", "close", "change", "return_pct", "open", "high", "low"]
    daily["date"] = pd.to_datetime(
        daily["date"], format="%Y/%m/%d", errors="coerce"
    )
    for column in daily.columns[1:]:
        daily[column] = pd.to_numeric(
            daily[column].astype(str).str.replace(",", "", regex=False),
            errors="coerce",
        )
    daily = daily.dropna(subset=["date", "close"]).set_index("date").sort_index()
    return daily[~daily.index.duplicated(keep="last")]


def load_daily_open_levels() -> pd.DataFrame:
    market = pd.read_csv(ROOT / "cache" / "market_daily.csv", parse_dates=["date"])
    with sqlite3.connect(get_path(RAW_DIR, "compass.db")) as connection:
        proxy = pd.read_sql(
            "select date, open, close from etf_prices where symbol = ? order by date",
            connection,
            params=("1028",),
        )
    proxy["date"] = pd.to_datetime(proxy["date"])
    proxy[["open", "close"]] = proxy[["open", "close"]].apply(
        pd.to_numeric, errors="coerce"
    )
    actual = market.loc[
        (market["symbol"] == "KODEX200")
        & market["open"].notna()
        & (market["date"] > pd.Timestamp("2009-03-31"))
    ].copy()
    first_actual = actual["date"].min()
    actual_anchor = float(actual.loc[actual["date"] == first_actual, "open"].iloc[0])
    nearest = proxy.iloc[(proxy["date"] - first_actual).abs().argsort()[:1]]
    proxy["open"] *= actual_anchor / float(nearest["open"].iloc[0])
    proxy = proxy.loc[proxy["date"] < first_actual]
    kodex = pd.concat(
        [proxy[["date", "open"]], actual[["date", "open"]]], ignore_index=True
    )
    kodex = (
        kodex.sort_values("date")
        .drop_duplicates("date", keep="last")
        .set_index("date")["open"]
    )

    bond = pd.read_csv(get_path(RAW_DIR, "krx_bond_index.csv"), encoding="cp949")
    bond.index = pd.to_datetime(bond.iloc[:, 0])
    bond_level = pd.to_numeric(
        bond.iloc[:, 1].astype(str).str.replace(",", "", regex=False), errors="coerce"
    ).rename("BOND")

    pivot_open = market.pivot_table(
        index="date", columns="symbol", values="open", aggfunc="last"
    ).sort_index()
    pivot_close = market.pivot_table(
        index="date", columns="symbol", values="close", aggfunc="last"
    ).sort_index()
    fx = pivot_close["USDKRW"].reindex(
        pd.date_range(pivot_close.index.min(), pivot_close.index.max(), freq="D")
    ).ffill()
    gld = (pivot_open["GLD"] * fx.reindex(pivot_open.index).to_numpy()).rename("GLD")
    uso = (pivot_open["USO"] * fx.reindex(pivot_open.index).to_numpy()).rename("USO")
    levels = pd.concat(
        [kodex.rename("KODEX200"), bond_level, gld, uso], axis=1, sort=False
    ).sort_index()
    calendar = pd.date_range(levels.index.min(), levels.index.max(), freq="B")
    return levels.reindex(calendar).ffill(limit=5)[ASSETS]


def daily_causal_percentile(series: pd.Series, window: int = 252, minimum: int = 126) -> pd.Series:
    def last_rank(values: np.ndarray) -> float:
        finite = values[np.isfinite(values)]
        if len(finite) < minimum:
            return np.nan
        return float((np.sum(finite[:-1] <= finite[-1]) + 1) / len(finite))

    return series.rolling(window + 1, min_periods=minimum).apply(last_rank, raw=True)


def build_daily_vkospi_signals() -> pd.DataFrame:
    daily = load_vkospi_daily()
    close = daily["close"]
    output = pd.DataFrame(index=daily.index)
    output["vkospi_close"] = close
    output["level_percentile"] = daily_causal_percentile(close)
    for window in (5, 10, 21):
        output[f"momentum_{window}"] = close.pct_change(window, fill_method=None)
    return output.replace([np.inf, -np.inf], np.nan)


def prepare_arrays(
    levels: pd.DataFrame,
    reference: pd.DataFrame,
    vkospi_signals: pd.DataFrame,
) -> dict[str, object]:
    forward = levels.shift(-1).div(levels).sub(1.0)
    dates = levels.index[
        (levels.index.to_period("M") >= reference.index.min())
        & (levels.index.to_period("M") <= reference.index.max())
    ]
    valid_dates: list[pd.Timestamp] = []
    returns: list[np.ndarray] = []
    base_weights: list[np.ndarray] = []
    months: list[pd.Period] = []
    level_percentile: list[float] = []
    momentum = {window: [] for window in (5, 10, 21)}
    signal_dates: list[pd.Timestamp | pd.NaT] = []
    weight_columns = [f"w_{asset}" for asset in ASSETS]

    for date in dates:
        month = date.to_period("M")
        if month not in reference.index:
            continue
        asset_return = forward.loc[date, ASSETS].to_numpy(dtype=float)
        if not np.isfinite(asset_return).all():
            continue
        cutoff = date - pd.Timedelta(days=1)
        known = vkospi_signals.loc[:cutoff]
        signal = known.iloc[-1] if not known.empty else pd.Series(dtype=float)
        valid_dates.append(date)
        returns.append(asset_return)
        base_weights.append(reference.loc[month, weight_columns].to_numpy(dtype=float))
        months.append(month)
        signal_dates.append(known.index[-1] if not known.empty else pd.NaT)
        level_percentile.append(float(signal.get("level_percentile", np.nan)))
        for window in momentum:
            momentum[window].append(float(signal.get(f"momentum_{window}", np.nan)))

    return {
        "dates": pd.DatetimeIndex(valid_dates),
        "returns": np.asarray(returns),
        "base_weights": np.asarray(base_weights),
        "months": pd.PeriodIndex(months, freq="M"),
        "signal_dates": pd.DatetimeIndex(signal_dates),
        "level_percentile": np.asarray(level_percentile),
        "momentum": {window: np.asarray(values) for window, values in momentum.items()},
    }


def simulate(
    arrays: dict[str, object],
    cfg: DynamicRiskConfig | None,
    start: pd.Period | None = None,
    end: pd.Period | None = None,
    cost_multiplier: float = 1.0,
    keep_daily: bool = True,
    stress_override: np.ndarray | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    dates = pd.DatetimeIndex(arrays["dates"])
    months = pd.PeriodIndex(arrays["months"], freq="M")
    mask = np.ones(len(dates), dtype=bool)
    if start is not None:
        mask &= months >= start
    if end is not None:
        mask &= months <= end
    positions = np.flatnonzero(mask)
    if stress_override is not None:
        stress = np.asarray(stress_override, dtype=float)
        if stress.shape != (len(dates),):
            raise ValueError(
                f"stress_override must have shape {(len(dates),)}, got {stress.shape}"
            )
        stress = np.nan_to_num(stress, nan=0.0, posinf=1.0, neginf=0.0).clip(0, 1)
    else:
        stress = np.zeros(len(dates)) if cfg is None else _stress_series(arrays, cfg)
    returns = np.asarray(arrays["returns"], dtype=float)
    base_weights = np.asarray(arrays["base_weights"], dtype=float)
    signal_dates = pd.DatetimeIndex(arrays["signal_dates"])

    pretrade = np.zeros(len(ASSETS))
    previous_month: pd.Period | None = None
    nav = 1.0
    peak = 1.0
    first_trade = True
    rf_daily = (1 + (cfg.financing_rate if cfg else 0.04)) ** (1 / 252) - 1
    rows: list[dict[str, object]] = []

    for position in positions:
        date = dates[position]
        month = months[position]
        base = base_weights[position].copy()
        severity = float(stress[position])
        transfer_fraction = 0.0 if cfg is None else cfg.max_risk_transfer * severity
        desired = base.copy()
        removed_equity = desired[0] * transfer_fraction
        removed_oil = desired[3] * transfer_fraction
        desired[0] -= removed_equity
        desired[3] -= removed_oil
        removed = removed_equity + removed_oil
        bond_share = 0.50 if cfg is None else cfg.bond_share
        desired[1] += removed * bond_share
        desired[2] += removed * (1 - bond_share)

        month_boundary = previous_month is None or month != previous_month
        band = math.inf if cfg is None else cfg.rebalance_band
        desired_turnover = 0.5 * float(np.abs(desired - pretrade).sum())
        rebalance = month_boundary or desired_turnover >= band
        weights = desired if rebalance else pretrade.copy()
        delta = weights - pretrade
        turnover = float(np.abs(delta).sum()) if first_trade else 0.5 * float(
            np.abs(delta).sum()
        )
        trade_cost = float(np.abs(delta).sum()) * 0.0015 * cost_multiplier
        fx_cost = (
            abs((weights[2] + weights[3]) - (pretrade[2] + pretrade[3]))
            * 0.0005
            * cost_multiplier
        )
        debt_weight = 1.0 - float(weights.sum())
        asset_return = returns[position]
        gross_return = float(weights @ asset_return + debt_weight * rf_daily)
        net_return = gross_return - trade_cost - fx_cost
        nav *= 1 + net_return
        peak = max(peak, nav)
        pretrade = weights * (1 + asset_return) / (1 + gross_return)
        rows.append(
            {
                "date": date,
                "month": month,
                "return": net_return,
                "gross_return": gross_return,
                "nav": nav,
                "drawdown": nav / peak - 1,
                "turnover": turnover,
                "trade_cost": trade_cost,
                "fx_cost": fx_cost,
                "stress": severity,
                "transfer_fraction": transfer_fraction,
                "signal_date": signal_dates[position],
                **{f"w_{asset}": weights[i] for i, asset in enumerate(ASSETS)},
            }
        )
        previous_month = month
        first_trade = False

    daily = pd.DataFrame(rows).set_index("date")
    monthly = daily.groupby("month").agg(
        return_factor=("return", lambda x: float(np.prod(1 + x))),
        gross_factor=("gross_return", lambda x: float(np.prod(1 + x))),
        turnover=("turnover", "sum"),
        trade_cost=("trade_cost", "sum"),
        fx_cost=("fx_cost", "sum"),
        avg_stress=("stress", "mean"),
        max_stress=("stress", "max"),
        avg_transfer=("transfer_fraction", "mean"),
    )
    monthly["return"] = monthly["return_factor"] - 1
    monthly["gross_return"] = monthly["gross_factor"] - 1
    wealth = (1 + monthly["return"]).cumprod()
    monthly["nav"] = wealth
    monthly["drawdown"] = wealth / wealth.cummax() - 1
    return (daily if keep_daily else pd.DataFrame()), monthly


def reconcile_to_monthly_reference(
    monthly_reference: pd.DataFrame,
    daily_baseline: pd.DataFrame,
    daily_overlay: pd.DataFrame,
) -> pd.DataFrame:
    """Apply only the daily overlay's relative return to the validated monthly path.

    The daily reconstruction differs slightly from the original monthly engine
    because financing and costs compound at a different frequency. Taking the
    overlay/base relative factor isolates the incremental VKOSPI allocation
    effect without pretending that this frequency-conversion gap is alpha.
    """
    common = (
        monthly_reference.index.intersection(daily_baseline.index)
        .intersection(daily_overlay.index)
        .sort_values()
    )
    relative_factor = (1 + daily_overlay.loc[common, "return"]).div(
        1 + daily_baseline.loc[common, "return"]
    )
    output = pd.DataFrame(index=common)
    output["reference_return"] = monthly_reference.loc[common, "return"]
    output["overlay_relative_return"] = relative_factor - 1
    output["return"] = (1 + output["reference_return"]) * relative_factor - 1
    output["gross_return"] = output["return"]
    output["turnover"] = daily_overlay.loc[common, "turnover"]
    output["trade_cost"] = daily_overlay.loc[common, "trade_cost"]
    output["fx_cost"] = daily_overlay.loc[common, "fx_cost"]
    output["avg_stress"] = daily_overlay.loc[common, "avg_stress"]
    output["max_stress"] = daily_overlay.loc[common, "max_stress"]
    output["avg_transfer"] = daily_overlay.loc[common, "avg_transfer"]
    wealth = (1 + output["return"]).cumprod()
    output["nav"] = wealth
    output["drawdown"] = wealth / wealth.cummax() - 1
    return output


## 10. 모듈 H — Robust VKOSPI 특징과 스트레스

VKOSPI 수준은 126·252일 인과적 백분위와 median/MAD z-score,
충격은 5·10·21일 로그변화를 직전 63일 변동성으로 나눠 만듭니다.
배포 설정 acceleration은 수준 40%, 5일 충격 35%, 5일 가속도
25%입니다. 최대 35%의 주식·원유 비중을 줄여 전부 금으로 옮깁니다.


In [ ]:
@dataclass(frozen=True)
class RobustStressConfig:
    mode: str
    level_threshold: float
    shock_threshold: float
    max_risk_transfer: float
    bond_share: float
    rebalance_band: float
    financing_rate: float = 0.04

    @property
    def name(self) -> str:
        return (
            f"{self.mode}_lt{self.level_threshold:.2f}_st{self.shock_threshold:.2f}"
            f"_rt{self.max_risk_transfer:.2f}_bs{self.bond_share:.2f}"
            f"_rb{self.rebalance_band:.2f}"
        )

    def dynamic_config(self) -> DynamicRiskConfig:
        return DynamicRiskConfig(
            mode="level",
            level_threshold=self.level_threshold,
            momentum_window=5,
            spike_threshold=0.0,
            max_risk_transfer=self.max_risk_transfer,
            bond_share=self.bond_share,
            rebalance_band=self.rebalance_band,
            financing_rate=self.financing_rate,
        )


def robust_zscore(series: pd.Series, window: int, minimum: int) -> pd.Series:
    median = series.rolling(window, min_periods=minimum).median()
    mad = (series - median).abs().rolling(window, min_periods=minimum).median()
    return ((series - median) / (1.4826 * mad.replace(0, np.nan))).clip(-6, 6)


def build_robust_daily_features() -> pd.DataFrame:
    daily = load_vkospi_daily()
    close = daily["close"].astype(float)
    log_close = np.log(close.where(close > 0))
    log_return = log_close.diff()
    output = pd.DataFrame(index=daily.index)
    output["close"] = close
    output["percentile_126"] = daily_causal_percentile(close, window=126, minimum=84)
    output["percentile_252"] = daily_causal_percentile(close, window=252, minimum=126)
    output["robust_z_63"] = robust_zscore(log_close, 63, 42)
    output["robust_z_252"] = robust_zscore(log_close, 252, 126)
    for window in (5, 10, 21):
        scale = log_return.rolling(63, min_periods=42).std(ddof=1) * math.sqrt(window)
        output[f"shock_{window}"] = (
            log_close.diff(window) / scale.replace(0, np.nan)
        ).clip(-8, 8)
    output["acceleration_5"] = log_close.diff(5) - log_close.diff(5).shift(5)
    acceleration_scale = log_return.rolling(63, min_periods=42).std(ddof=1) * math.sqrt(5)
    output["acceleration_z5"] = (
        output["acceleration_5"] / acceleration_scale.replace(0, np.nan)
    ).clip(-8, 8)

    high = daily["high"].astype(float).where(daily["high"].astype(float) > 0, close)
    low = daily["low"].astype(float).where(daily["low"].astype(float) > 0, close)
    high21 = high.rolling(21, min_periods=10).max()
    low21 = low.rolling(21, min_periods=10).min()
    output["distance_high21"] = close / high21 - 1
    output["close_location21"] = (
        (close - low21) / (high21 - low21).replace(0, np.nan)
    ).clip(0, 1)
    output["positive_fraction5"] = (log_return > 0).rolling(5, min_periods=3).mean()
    output["positive_fraction21"] = (log_return > 0).rolling(21, min_periods=10).mean()
    output["fast_slow"] = log_close.diff(5) - 5 / 21 * log_close.diff(21)
    return output.replace([np.inf, -np.inf], np.nan)


def align_features_to_arrays(
    features: pd.DataFrame, arrays: dict[str, object]
) -> pd.DataFrame:
    signal_dates = pd.DatetimeIndex(arrays["signal_dates"])
    aligned = features.reindex(signal_dates)
    aligned.index = pd.RangeIndex(len(aligned))
    return aligned


def stress_from_features(
    aligned: pd.DataFrame,
    mode: str,
    level_threshold: float,
    shock_threshold: float,
) -> np.ndarray:
    percentile = aligned["percentile_252"].fillna(aligned["percentile_126"])
    level = np.clip(
        (percentile.to_numpy(dtype=float) - level_threshold)
        / max(1 - level_threshold, 1e-6),
        0,
        1,
    )
    shock_raw = aligned["shock_5"].to_numpy(dtype=float)
    shock = np.clip((shock_raw - shock_threshold) / 2.5, 0, 1)
    acceleration = np.clip(
        (aligned["acceleration_z5"].to_numpy(dtype=float) - shock_threshold) / 2.5,
        0,
        1,
    )
    confirmation = np.nan_to_num(
        aligned["close_location21"].to_numpy(dtype=float), nan=0.5
    ).clip(0, 1)
    distance = np.nan_to_num(
        aligned["distance_high21"].to_numpy(dtype=float), nan=0.0
    )
    falling = np.clip(-shock_raw / 2.5, 0, 1)
    exhaustion = np.clip(level * (-distance).clip(0, 0.5) / 0.5 * falling, 0, 1)
    if mode == "robust_mean":
        stress = 0.50 * level + 0.50 * shock
    elif mode == "robust_max":
        stress = np.maximum(level, shock)
    elif mode == "confirmed":
        stress = (0.45 * level + 0.55 * shock) * (0.35 + 0.65 * confirmation)
    elif mode == "acceleration":
        stress = 0.40 * level + 0.35 * shock + 0.25 * acceleration
    elif mode == "exhaustion_adjusted":
        stress = (0.50 * level + 0.50 * shock) * (1 - 0.75 * exhaustion)
    else:
        raise ValueError(mode)
    return np.nan_to_num(stress, nan=0.0, posinf=1.0, neginf=0.0).clip(0, 1)


In [ ]:
ROBUST_CONFIG = RobustStressConfig(
    mode="acceleration",
    level_threshold=0.90,
    shock_threshold=1.00,
    max_risk_transfer=0.35,
    bond_share=0.00,
    rebalance_band=0.20,
    financing_rate=0.04,
)


def run_robust_vkospi_overlay(reference: pd.DataFrame):
    levels = load_daily_open_levels()
    arrays = prepare_arrays(
        levels, reference, build_daily_vkospi_signals()
    )
    features = align_features_to_arrays(
        build_robust_daily_features(), arrays
    )
    stress = stress_from_features(
        features,
        ROBUST_CONFIG.mode,
        ROBUST_CONFIG.level_threshold,
        ROBUST_CONFIG.shock_threshold,
    )
    _, neutral_monthly = simulate(
        arrays, None, keep_daily=False
    )
    daily, overlay_monthly = simulate(
        arrays,
        ROBUST_CONFIG.dynamic_config(),
        keep_daily=True,
        stress_override=stress,
    )
    reconciled = reconcile_to_monthly_reference(
        reference, neutral_monthly, overlay_monthly
    )
    valid = daily["signal_date"].notna()
    assert (
        daily.index[valid].to_numpy()
        > pd.DatetimeIndex(
            daily.loc[valid, "signal_date"]
        ).to_numpy()
    ).all()
    return daily, overlay_monthly, reconciled


final_daily, overlay_monthly, final = (
    run_robust_vkospi_overlay(medium)
)
print(
    "최종 월별 경로:",
    final.index.min(), "→", final.index.max(), len(final)
)
display(pd.Series(asdict(ROBUST_CONFIG), name="value").to_frame())


## 11. 성과와 누수 방지 검증

전체기간과 2018년 이후 잠금구간을 따로 봅니다. 기대값은 결과파일에서
읽는 값이 아니라 공개형 계산이 기준전략과 같은지 검사하는 회귀 테스트
상수입니다. 데이터나 로직이 바뀌면 검증은 의도적으로 실패합니다.


In [ ]:
def period_metrics(path: pd.DataFrame, start: str | None = None):
    view = (
        path
        if start is None
        else path.loc[pd.Period(start, "M"):]
    )
    return performance_summary(view["return"])


metrics = pd.concat(
    {
        "Full_2007_2026": period_metrics(final),
        "Locked_2018_2026": period_metrics(
            final, CONFIG.locked_start
        ),
    },
    axis=1,
).T
display(metrics[[
    "Months", "CAGR", "Sharpe", "MDD",
    "Calmar", "FinalMultiple",
]])

EXPECTED = {
    "Full_2007_2026": {
        "CAGR": 0.15636553973863565,
        "Sharpe": 1.1325286822772858,
        "MDD": -0.12958799769553853,
    },
    "Locked_2018_2026": {
        "CAGR": 0.20907823332988906,
        "Sharpe": 1.4969186464577038,
        "MDD": -0.09760421330851343,
    },
}
for period, expected in EXPECTED.items():
    for metric, value in expected.items():
        actual = float(metrics.loc[period, metric])
        assert np.isclose(
            actual, value, atol=5e-11, rtol=0
        ), (period, metric, actual, value)

assert (signals["signal_month"] < signals.index).all()
assert final.index.equals(medium.index)
assert final["return"].notna().all()
print(
    "PASS: 기준 성과, 월간 정렬, 거시 1개월 시차, "
    "VKOSPI 전일 시차 확인"
)


In [ ]:
wealth = pd.DataFrame(
    {
        "SLSQP base": (1 + defensive["return"]).cumprod(),
        "Tail + vol target": (1 + medium["return"]).cumprod(),
        "Robust VKOSPI final": (1 + final["return"]).cumprod(),
    }
)
ax = wealth.plot(figsize=(12, 5), logy=True, grid=True)
ax.set_title("Robust VKOSPI reference strategy")
ax.set_ylabel("Wealth multiple (log scale)")
plt.show()


## 12. 결과 내보내기

월별 수익률·비중, 거시확률, 16개 입력패널, 로지스틱 확률,
일간 VKOSPI 거래경로와 성과표를 CSV/JSON으로 저장하고 ZIP으로
묶습니다.


In [ ]:
signals.to_csv(OUTPUT_DIR / "macro_signals.csv")
macro_probabilities.to_csv(
    OUTPUT_DIR / "macro_probabilities.csv"
)
model_data.to_csv(OUTPUT_DIR / "logistic_input_panel.csv")
factor.to_csv(OUTPUT_DIR / "tail_factor.csv")
medium.to_csv(OUTPUT_DIR / "medium_monthly.csv")
final.to_csv(OUTPUT_DIR / "final_reconciled_monthly.csv")
final_daily.to_csv(OUTPUT_DIR / "robust_vkospi_daily.csv")
metrics.to_csv(OUTPUT_DIR / "performance_summary.csv")
(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(
        {
            "notebook": asdict(CONFIG),
            "robust_vkospi": asdict(ROBUST_CONFIG),
            "tail_features": TAIL_FEATURES,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

archive = shutil.make_archive(
    str(ROOT / "robust_vkospi_results"), "zip", OUTPUT_DIR
)
print("저장 완료:", archive)
if IN_COLAB:
    files.download(archive)


## 해석할 때 주의할 점

이 노트북은 재현용 기준전략입니다. 2018년 이후 구간도 전략 연구
과정에서 이미 관찰됐으므로 완전히 새로운 미사용 표본으로 해석하면
안 됩니다. 거래비용은 구현값을 반영하지만 세금, 추적오차, 실제
호가충격은 별도입니다. OAP 4개 합성치는 ZIP에 이미 월별 입력으로
들어 있으며 원 논문의 개별 신호를 다시 내려받는 구조는 아닙니다.
